In [ ]:
import torch
from mmcv.cnn import get_model_complexity_info
from mmocr.models import build_detector
from mmcv import Config
import numpy as np

In [ ]:
# Load config
config_path = "work_dirs/totaltext/kac4/lranet_totaltext_det_single_head.py"
cfg = Config.fromfile(config_path)
cfg.model.train_cfg = None
cfg.model.test_cfg = None

# Build detector
model = build_detector(cfg.model, test_cfg=cfg.get('test_cfg'))
model.eval()
model.cuda()

# Create wrapper to call model with return_loss=False
class InferenceWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, img, img_metas):
        with torch.no_grad():
            return self.model(img=img, img_metas=img_metas, return_loss=False)

wrapped_model = InferenceWrapper(model)

# Construct dummy input with img_metas

def input_constructor(input_shape):
    C, H, W = input_shape
    dummy_img = torch.randn(1, C, H, W).cuda()

    dummy_meta = {
        'img_shape': (H, W, C),
        'ori_shape': (H, W, C),
        'pad_shape': (H, W, C),
        'flip': False,
        'batch_input_shape': (H, W),
        'scale_factor': np.array([1.0, 1.0, 1.0, 1.0], dtype=np.float32),  # 👈 this is the fix
    }

    return dict(img=[dummy_img], img_metas=[[dummy_meta]])



# Your test resolution: (channels, height, width)
input_shape = (3, 1000, 1800)

# Run complexity analysis
flops, params = get_model_complexity_info(
    wrapped_model,
    input_shape=input_shape,
    input_constructor=input_constructor,
    as_strings=True,
    print_per_layer_stat=False,
)

print(f'FLOPs: {flops}')
print(f'Params: {params}')
